# V2 Evaluation

Comprehensive evaluation of all V2 models on the MSR-VTT 1K test set.

**Models evaluated:**
1. CLIP Single-Frame Zero-Shot (direct retrieval, no training — true baseline)
2. MLP Cosine V2 (normalized output, 50 epochs)
3. MLP InfoNCE V2 (contrastive, 50 epochs)
4. Transformer Fusion V2 (5-token, joint loss, 50 epochs)

**Metrics:** Cosine Similarity, MSE, R@1, R@5, R@10, MedR, Latency

**Visualizations:** Joint embedding PCA, similarity matrices, retrieval bar chart, latency comparison

## Step 1: Imports & Setup

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
os.makedirs("eval_v2", exist_ok=True)

## Step 2: Dataset & Model Classes

In [ ]:
class MultimodalDatasetV2(Dataset):
    def __init__(self, file_path):
        data = torch.load(file_path, map_location="cpu")
        self.z_img     = data["z_img"].float()
        self.z_aud     = data["z_aud"].float()
        self.v_teacher = data["v_teacher"].float()
        self.has_audio = data.get("has_audio", torch.ones(len(self.z_img), dtype=torch.bool))
        print(f"Test set: {len(self.z_img)} videos | audio: {self.has_audio.sum().item()}")

    def __len__(self):
        return len(self.z_img)

    def __getitem__(self, idx):
        return {"z_img": self.z_img[idx], "z_aud": self.z_aud[idx], "v_teacher": self.v_teacher[idx]}


class MLPApproximatorV2(nn.Module):
    def __init__(self, hidden_dims=(1024, 2048, 1024), output_dim=1024, dropout=0.1, normalize_output=False):
        super().__init__()
        self.normalize_output = normalize_output
        input_dim = 3 * 512 + 128
        layers, in_d = [], input_dim
        for h_d in hidden_dims:
            layers += [nn.Linear(in_d, h_d), nn.BatchNorm1d(h_d), nn.GELU(), nn.Dropout(dropout)]
            in_d = h_d
        layers.append(nn.Linear(in_d, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, z_img, z_aud):
        x = torch.cat([z_img.reshape(z_img.size(0), -1), z_aud], dim=-1)
        out = self.network(x)
        return F.normalize(out, dim=-1) if self.normalize_output else out


class TransformerFusionV2(nn.Module):
    def __init__(self, img_dim=512, aud_dim=128, embed_dim=384, num_heads=6,
                 num_layers=3, output_dim=1024, dropout=0.1, num_frames=3):
        super().__init__()
        self.num_frames = num_frames
        self.img_proj      = nn.Linear(img_dim, embed_dim)
        self.aud_proj      = nn.Linear(aud_dim, embed_dim)
        self.cls_token     = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.modality_embed = nn.Embedding(3, embed_dim)
        self.pos_embed     = nn.Parameter(torch.zeros(1, 1 + num_frames + 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, embed_dim * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 2, output_dim))

    def forward(self, z_img, z_aud):
        B = z_img.size(0)
        img_tokens = self.img_proj(z_img)
        aud_token  = self.aud_proj(z_aud).unsqueeze(1)
        cls_token  = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_token, img_tokens, aud_token], dim=1)
        type_ids = torch.tensor([0] + [1]*self.num_frames + [2], dtype=torch.long, device=z_img.device)
        tokens = tokens + self.modality_embed(type_ids) + self.pos_embed
        return self.head(self.transformer(tokens)[:, 0])


print("Classes defined.")

## Step 3: Load Data & Models

In [ ]:
def find_file(name, search_dirs=None):
    dirs = search_dirs or [".", "models_v2", "../models_v2",
                           "models_v2/mlp_cosine_v2", "models_v2/mlp_infonce_v2",
                           "models_v2/transformer_v2"]
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    # Recursive search
    for root, _, files in os.walk("."):
        for f in files:
            if f == name:
                return os.path.join(root, f)
    raise FileNotFoundError(f"{name} not found.")

# Load test set
test_ds = MultimodalDatasetV2(find_file("test_features_v2.pt", [".", "/kaggle/working",
                                                                  "../features"]))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# Load models (try best checkpoint, fall back to final)
def load_model(model, basename):
    for suffix in ["_best.pt", "_final.pt"]:
        try:
            path = find_file(basename + suffix)
            model.load_state_dict(torch.load(path, map_location=device))
            print(f"Loaded {path}")
            return model
        except FileNotFoundError:
            continue
    print(f"WARNING: No checkpoint found for {basename} — using random weights.")
    return model

mlp_cosine  = load_model(MLPApproximatorV2(normalize_output=True).to(device),  "mlp_cosine_v2")
mlp_infonce = load_model(MLPApproximatorV2(normalize_output=False).to(device), "mlp_infonce_v2")
transformer = load_model(TransformerFusionV2().to(device),                      "transformer_v2")

for m in [mlp_cosine, mlp_infonce, transformer]:
    m.eval()

## Step 4: Retrieval Metric Computation

In [ ]:
def compute_retrieval_metrics(pred_n, target_n):
    """
    pred_n, target_n: L2-normalized [N, D] tensors.
    For each query i, rank of its correct match (rank 1 = perfect).
    Returns dict with R@1, R@5, R@10, MedR.
    """
    sim_mat = torch.matmul(pred_n, target_n.T)  # [N, N]
    N = sim_mat.size(0)
    ranks = torch.tensor(
        [(sim_mat[i] > sim_mat[i, i]).sum().item() + 1 for i in range(N)],
        dtype=torch.float
    )
    return {
        "r1":   (ranks <= 1).float().mean().item()  * 100,
        "r5":   (ranks <= 5).float().mean().item()  * 100,
        "r10":  (ranks <= 10).float().mean().item() * 100,
        "medr": ranks.median().item(),
    }


def evaluate(model, loader, model_name, n_latency_runs=200):
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            z_img = batch["z_img"].to(device)
            z_aud = batch["z_aud"].to(device)
            pred  = model(z_img, z_aud)
            preds.append(pred.cpu())
            targets.append(batch["v_teacher"])
    preds   = torch.cat(preds)
    targets = torch.cat(targets)

    pred_n   = F.normalize(preds,   dim=-1)
    target_n = F.normalize(targets, dim=-1)
    cos_sim  = (pred_n * target_n).sum(-1).mean().item()
    mse      = F.mse_loss(preds, targets).item()
    retrieval = compute_retrieval_metrics(pred_n, target_n)

    # Single-sample latency (CPU, same as V1 for fair comparison)
    dummy_img = test_ds.z_img[:1].to("cpu")
    dummy_aud = test_ds.z_aud[:1].to("cpu")
    m_cpu = model.cpu()
    m_cpu.eval()
    # Warmup
    for _ in range(20):
        with torch.no_grad():
            m_cpu(dummy_img, dummy_aud)
    start = time.perf_counter()
    for _ in range(n_latency_runs):
        with torch.no_grad():
            m_cpu(dummy_img, dummy_aud)
    latency_ms = (time.perf_counter() - start) / n_latency_runs * 1000
    model.to(device)

    result = {"name": model_name, "cosine": cos_sim, "mse": mse,
              "latency_ms": latency_ms, **retrieval,
              "preds": preds, "targets": targets}
    print(f"\n{model_name}")
    print(f"  Cosine Sim  : {cos_sim:.4f}")
    print(f"  MSE         : {mse:.4f}")
    print(f"  R@1/5/10    : {retrieval['r1']:.1f}% / {retrieval['r5']:.1f}% / {retrieval['r10']:.1f}%")
    print(f"  MedR        : {retrieval['medr']:.0f}")
    print(f"  Latency     : {latency_ms:.2f} ms/sample (CPU)")
    return result


print("Evaluation function ready.")

## Step 5: CLIP Zero-Shot Baseline

Directly use the middle CLIP frame embedding (z_img[:, 1, :]) projected to 1024-dim via a random (untrained) linear layer and as-is via cosine retrieval.

This is the true zero-shot baseline: how well can a single CLIP frame retrieve the correct ImageBind embedding with NO learned mapping?

In [ ]:
# Collect all test features
all_z_img = test_ds.z_img        # [N, 3, 512]
all_targets = test_ds.v_teacher  # [N, 1024]

target_n = F.normalize(all_targets, dim=-1)

# Zero-shot: use middle frame (index 1) directly
mid_frame = all_z_img[:, 1, :]     # [N, 512]
mid_frame_n = F.normalize(mid_frame.float(), dim=-1)  # [N, 512]

# Cosine retrieval in CLIP space vs ImageBind teacher
# (different dims — can only compute if we project; skip cosine, do retrieval via sim)
# We pad to 1024 with zeros or use a random linear to compare fairly
# Best fair comparison: project 512 → 1024 via random linear (what untrained MLP would do)
torch.manual_seed(0)
random_proj = nn.Linear(512, 1024, bias=False)
with torch.no_grad():
    zero_shot_pred = random_proj(mid_frame)  # [N, 1024]
zero_shot_n = F.normalize(zero_shot_pred, dim=-1)

zs_retrieval = compute_retrieval_metrics(zero_shot_n, target_n)
zs_cos = (zero_shot_n * target_n).sum(-1).mean().item()

print("=== CLIP Single-Frame Zero-Shot (Random Projection) ===")
print(f"  Cosine Sim : {zs_cos:.4f}")
print(f"  R@1/5/10   : {zs_retrieval['r1']:.1f}% / {zs_retrieval['r5']:.1f}% / {zs_retrieval['r10']:.1f}%")
print(f"  MedR       : {zs_retrieval['medr']:.0f}")
print("  (Expected ~0.1% R@1 for 1000-way random retrieval)")

zero_shot_result = {"name": "CLIP Zero-Shot (random proj)", "cosine": zs_cos,
                    "mse": float("nan"), "latency_ms": 0.0,
                    **zs_retrieval, "preds": zero_shot_pred, "targets": all_targets}

## Step 6: Evaluate All Models

In [ ]:
results = [zero_shot_result]
results.append(evaluate(mlp_cosine,  test_loader, "MLP Cosine V2"))
results.append(evaluate(mlp_infonce, test_loader, "MLP InfoNCE V2"))
results.append(evaluate(transformer, test_loader, "Transformer V2"))

## Step 7: Results Table

In [ ]:
print("\n" + "="*85)
print(f"{'Model':<32} {'CosSim':>7} {'MSE':>7} {'R@1':>6} {'R@5':>6} {'R@10':>6} {'MedR':>6} {'ms/samp':>8}")
print("-"*85)
for r in results:
    mse_str = f"{r['mse']:7.4f}" if not (isinstance(r['mse'], float) and r['mse'] != r['mse']) else "    N/A"
    print(f"{r['name']:<32} {r['cosine']:7.4f} {mse_str} {r['r1']:6.1f} {r['r5']:6.1f} {r['r10']:6.1f} {r['medr']:6.0f} {r['latency_ms']:8.2f}")
print("="*85)

## Step 8: Recall@K Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names    = [r["name"] for r in results]
short_n  = [n.replace(" V2", "") for n in names]
colors   = ["#9E9E9E", "#2196F3", "#FF9800", "#4CAF50"]

# Recall@K grouped bar
k_vals = ["R@1", "R@5", "R@10"]
x = np.arange(len(k_vals))
width = 0.2
ax = axes[0]
for i, (r, c) in enumerate(zip(results, colors)):
    vals = [r["r1"], r["r5"], r["r10"]]
    ax.bar(x + i * width, vals, width, label=short_n[i], color=c, alpha=0.85)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(k_vals)
ax.set_ylabel("Recall (%)")
ax.set_title("Recall@K — V2 Models")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# MedR bar
ax2 = axes[1]
medr_vals = [r["medr"] for r in results]
ax2.bar(short_n, medr_vals, color=colors, alpha=0.85)
ax2.set_ylabel("Median Rank (lower = better)")
ax2.set_title("Median Rank — V2 Models")
ax2.set_xticklabels(short_n, rotation=15, ha="right")
ax2.grid(axis="y", alpha=0.3)
for i, v in enumerate(medr_vals):
    ax2.text(i, v + 5, f"{v:.0f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("eval_v2/retrieval_comparison_v2.png", dpi=150)
plt.show()
print("Saved retrieval_comparison_v2.png")

## Step 9: Joint Embedding Space PCA

Visualize all 4 embedding types in the same PCA space (whitened, variance-normalized — fixes the V1 PCA issue where raw VGGish 0-255 scale dominated axes).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

N_VIZ = min(500, len(test_ds))

# Gather embeddings for visualization
z_img_mid = test_ds.z_img[:N_VIZ, 1, :]     # middle frame CLIP [N, 512]
z_aud     = test_ds.z_aud[:N_VIZ]            # VGGish [N, 128]
v_teacher = test_ds.v_teacher[:N_VIZ]        # IB teacher [N, 1024]

# Get transformer predictions
transformer.eval()
with torch.no_grad():
    v_pred = transformer(
        test_ds.z_img[:N_VIZ].to(device),
        test_ds.z_aud[:N_VIZ].to(device)
    ).cpu()

def pca_2d_whitened(tensor):
    """Whitened (standardized) PCA — fixes the V1 scale issue."""
    arr = tensor.float().numpy()
    arr_scaled = StandardScaler().fit_transform(arr)
    return PCA(n_components=2).fit_transform(arr_scaled)

clip_2d    = pca_2d_whitened(F.normalize(z_img_mid, dim=-1))
vggish_2d  = pca_2d_whitened(z_aud)
teacher_2d = pca_2d_whitened(F.normalize(v_teacher, dim=-1))
pred_2d    = pca_2d_whitened(F.normalize(v_pred, dim=-1))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ["CLIP (mid frame)", "VGGish Audio", "IB Teacher", "Transformer V2 Pred"]
data2d = [clip_2d, vggish_2d, teacher_2d, pred_2d]
colors_pca = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

for ax, d, t, c in zip(axes, data2d, titles, colors_pca):
    ax.scatter(d[:, 0], d[:, 1], c=c, alpha=0.3, s=8, edgecolors="none")
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.2)

plt.suptitle("Whitened PCA (StandardScaler) — 500 test samples", fontsize=12)
plt.tight_layout()
plt.savefig("eval_v2/embedding_pca_v2.png", dpi=150)
plt.show()
print("Saved embedding_pca_v2.png")

## Step 10: Teacher vs Prediction Overlay

Put the teacher and the best model's prediction in the same PCA space — they should overlap if the approximation is good.

In [ ]:
# Fit PCA on teacher, project both teacher and pred into that space
from sklearn.decomposition import PCA as SKPCA

tch_arr  = F.normalize(v_teacher, dim=-1).float().numpy()
pred_arr = F.normalize(v_pred,    dim=-1).float().numpy()

# Fit PCA on teacher embeddings
pca = SKPCA(n_components=2)
tch_scaled  = StandardScaler().fit_transform(tch_arr)
pca.fit(tch_scaled)
tch_2d  = pca.transform(tch_scaled)

# Project pred using same scaler+PCA
from sklearn.preprocessing import StandardScaler as SS
scaler = SS().fit(tch_arr)
pred_2d_overlay = pca.transform(scaler.transform(pred_arr))

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(tch_2d[:, 0],         tch_2d[:, 1],         c="#4CAF50", alpha=0.4, s=10, label="IB Teacher")
ax.scatter(pred_2d_overlay[:, 0], pred_2d_overlay[:, 1], c="#E91E63", alpha=0.4, s=10, label="Transformer V2")
ax.set_title("Teacher vs Transformer V2 Prediction\n(same PCA space)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("eval_v2/teacher_vs_pred_overlay_v2.png", dpi=150)
plt.show()
print("Saved teacher_vs_pred_overlay_v2.png")

## Step 11: Cosine Similarity Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for r, c in zip(results[1:], colors[1:]):  # skip zero-shot
    pred_n_full   = F.normalize(r["preds"].float(),   dim=-1)
    target_n_full = F.normalize(r["targets"].float(), dim=-1)
    per_sample_cos = (pred_n_full * target_n_full).sum(-1).numpy()
    ax.hist(per_sample_cos, bins=50, alpha=0.5, label=r["name"].replace(" V2", ""), color=c)

ax.set_xlabel("Per-Sample Cosine Similarity")
ax.set_ylabel("Count")
ax.set_title("Distribution of Per-Sample Cosine Similarity (test set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("eval_v2/cosine_distribution_v2.png", dpi=150)
plt.show()
print("Saved cosine_distribution_v2.png")

## Step 12: Latency & Efficiency Comparison

ImageBind teacher generation: ~2,090 ms/video on T4 GPU (from extraction logs).
Breakdown: CLIP ~50 ms, VGGish+ffmpeg ~300 ms, ImageBind vision ~1,500 ms, ImageBind audio ~240 ms.

In [ ]:
# Reference: ImageBind full pipeline timing from V2 extraction
# (Vision 16-frame + Audio encoding, T4 GPU)
IB_VISION_MS  = 1500.0  # ms — ImageBind video encoder
IB_AUDIO_MS   =  240.0  # ms — ImageBind audio encoder (new in V2)
CLIP_MS       =   50.0  # ms — CLIP frame extraction
VGGISH_MS     =  300.0  # ms — VGGish + ffmpeg
IB_TOTAL_MS   = IB_VISION_MS + IB_AUDIO_MS + CLIP_MS + VGGISH_MS  # 2090 ms

# Student pipeline (CPU): CLIP + VGGish + our model
for r in results[1:]:
    student_total = CLIP_MS + VGGISH_MS + r["latency_ms"]
    speedup = IB_TOTAL_MS / student_total
    print(f"{r['name']:<24}: model={r['latency_ms']:.2f}ms  "
          f"full-pipeline={student_total:.0f}ms  speedup={speedup:.0f}x vs IB")

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
model_names = [r["name"].replace(" V2", "") for r in results[1:]]
clip_part   = [CLIP_MS] * len(model_names)
vggish_part = [VGGISH_MS] * len(model_names)
model_part  = [r["latency_ms"] for r in results[1:]]

x = np.arange(len(model_names))
ax.bar(x, clip_part,   label="CLIP (3 frames)", color="#2196F3")
ax.bar(x, vggish_part, bottom=clip_part, label="VGGish+ffmpeg", color="#FF9800")
ax.bar(x, model_part,  bottom=[a+b for a,b in zip(clip_part, vggish_part)],
       label="Our Model", color="#4CAF50")

# ImageBind reference line
ax.axhline(IB_TOTAL_MS, color="red", linestyle="--", linewidth=2,
           label=f"ImageBind V2 pipeline ({IB_TOTAL_MS:.0f} ms)")
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha="right")
ax.set_ylabel("Latency (ms / video)")
ax.set_title("End-to-End Pipeline Latency (Student CPU vs ImageBind GPU)")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("eval_v2/latency_comparison_v2.png", dpi=150)
plt.show()
print("Saved latency_comparison_v2.png")

## Step 13: Save Results CSV

In [ ]:
import csv

csv_path = "eval_v2/accuracy_results_v2.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Model", "Cosine_Sim", "MSE", "R@1", "R@5", "R@10", "MedR", "Latency_ms"])
    for r in results:
        writer.writerow([
            r["name"],
            f"{r['cosine']:.6f}",
            f"{r['mse']:.6f}" if r['mse'] == r['mse'] else "nan",  # nan check
            f"{r['r1']:.2f}",
            f"{r['r5']:.2f}",
            f"{r['r10']:.2f}",
            f"{r['medr']:.0f}",
            f"{r['latency_ms']:.4f}",
        ])

print(f"Results saved to {csv_path}")

# Print final summary
print("\n===== V2 FINAL RESULTS =====")
for r in results:
    print(f"  {r['name']}: R@1={r['r1']:.1f}% | MedR={r['medr']:.0f} | cos={r['cosine']:.3f}")